In [3]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import json
from pathlib import Path
from sklearn.model_selection import train_test_split,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,OrdinalEncoder,PowerTransformer
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor ,StackingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [4]:
root_dir = Path.cwd().parent
data_dir = root_dir / 'data' / 'interim' / 'urbaneats-cleaned-dataset.csv'

In [5]:
df = pd.read_csv(data_dir)

In [6]:
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,pickup_time_minutes,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,INDO,19,3,Saturday,1,11.0,15.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,BANG,25,3,Friday,0,19.0,5.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,BANG,19,3,Saturday,1,8.0,15.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,COIMB,5,4,Tuesday,0,18.0,10.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,CHEN,26,3,Saturday,1,13.0,15.0,afternoon,6.210138,medium


In [7]:
df.shape

(45502, 27)

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

In [10]:
# check for missing values

df.isna().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [11]:
dagshub.init(repo_owner='AvanindraBose', repo_name='Urban-Eats-Food-Delivery-Time-Prediction', mlflow=True)

Accessing as AvanindraBose

Initialized MLflow to track repo "AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction"

Repository AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction initialized!

In [12]:
mlflow.set_tracking_uri('https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow')

# Droping Missing Values and then Final training the Best Stacking Regressor which will be used for inference.

In [13]:
temp_df = df.copy().dropna()

In [14]:
temp_df.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [15]:
temp_df.shape

(37695, 16)

In [16]:
X = temp_df.drop(columns= ['time_taken'])
y = temp_df['time_taken']

In [17]:
X.sample(10)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
36944,21.0,5.0,sandstorms,high,0,snack,motorcycle,0.0,no,metropolitian,0,5.0,afternoon,5.958821,medium
6650,27.0,5.0,windy,jam,1,drinks,scooter,0.0,no,urban,0,15.0,night,13.969606,long
178,38.0,4.7,windy,high,2,buffet,motorcycle,1.0,no,metropolitian,0,10.0,afternoon,6.217884,medium
36720,34.0,5.0,cloudy,jam,0,drinks,motorcycle,1.0,no,urban,0,15.0,evening,9.192290,medium
27888,32.0,4.9,stormy,low,1,meal,scooter,1.0,no,metropolitian,0,5.0,night,8.927644,medium
4188,34.0,4.7,cloudy,low,0,buffet,motorcycle,1.0,no,metropolitian,0,5.0,night,19.912967,very_long
17168,20.0,4.7,fog,jam,1,drinks,motorcycle,1.0,no,metropolitian,0,10.0,night,9.074331,medium
9636,31.0,4.6,sandstorms,medium,0,buffet,motorcycle,1.0,no,metropolitian,0,15.0,afternoon,6.128804,medium
13151,33.0,4.2,stormy,jam,1,meal,scooter,1.0,no,metropolitian,0,5.0,evening,7.562980,medium
4226,31.0,4.5,windy,low,0,buffet,motorcycle,0.0,no,metropolitian,1,15.0,night,4.537544,short


In [18]:
y.sample(10)

6404     12
7787     16
19319    48
1321     25
40190    18
34591    23
16507    16
6287     19
34808    26
15195    17
Name: time_taken, dtype: int64

In [19]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [20]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [21]:
temp_df.dtypes

age                    float64
ratings                float64
weather                 object
traffic                 object
vehicle_condition        int64
type_of_order           object
type_of_vehicle         object
multiple_deliveries    float64
festival                object
city_type               object
time_taken               int64
is_weekend               int64
pickup_time_minutes    float64
order_time_of_day       object
distance               float64
distance_type           object
dtype: object

In [22]:
num_cols = X_train.select_dtypes(include=np.number).columns.to_list()

In [23]:
num_cols.remove('vehicle_condition')
num_cols.remove('multiple_deliveries')

In [24]:
num_cols

['age', 'ratings', 'is_weekend', 'pickup_time_minutes', 'distance']

In [25]:
X_train.select_dtypes(include=object).columns.to_list()

['weather',
 'traffic',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'order_time_of_day',
 'distance_type']

In [26]:
ordinal_cat_cols = ['traffic','distance_type']

nominal_cat_cols = [
    'weather',
    'type_of_order',
    'type_of_vehicle',
    'festival',
    'city_type',
    'order_time_of_day'
]

In [27]:
len(num_cols + nominal_cat_cols + ordinal_cat_cols)

13

In [28]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

__Testing the Pipeline with Rf and XGB Regressor.__

In [29]:
run_id = 'b49702f40cbd4581a53f8b131c566df7'
artifact_path = 'xgb_best_params.json'
artifact_uri = f"runs:/{run_id}/{artifact_path}"

local_path = mlflow.artifacts.download_artifacts(artifact_uri, dst_path="artifacts")

In [30]:
with open(local_path,'r') as f :
    xgb_params = json.load(f)

In [31]:
run_id = 'c9c3f08094bb4b70983f057e293ed81a'
artifact_path = 'rf_best_params.json'
artifact_uri = f"runs:/{run_id}/{artifact_path}"

local_path = mlflow.artifacts.download_artifacts(artifact_uri, dst_path="artifacts")

In [32]:
with open(local_path,'r') as f :
    rf_params = json.load(f)

In [33]:
xgb_params

{'n_estimators': 603,
 'max_depth': 11,
 'learning_rate': 0.03417913351080966,
 'min_child_weight': 4,
 'gamma': 0.452621135308464,
 'subsample': 0.863443370509037,
 'colsample_bytree': 0.9651511117267577,
 'colsample_bylevel': 0.8881626404741021,
 'colsample_bynode': 0.7870334195129309,
 'reg_alpha': 8.213030683468667e-08,
 'reg_lambda': 0.2436995143218315,
 'max_delta_step': 1,
 'grow_policy': 'lossguide',
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1}

In [34]:
rf_params

{'n_estimators': 190,
 'max_depth': 15,
 'max_features': None,
 'min_samples_split': 10,
 'min_samples_leaf': 1,
 'max_samples': 0.8689643457612419,
 'random_state': 42,
 'n_jobs': -1}

# Lets Train the Final Estimator which will be used for inferencing.

In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
    )

stack = StackingRegressor(
        estimators=[
            ("xgb", XGBRegressor(**xgb_params)),
            ("rf", RandomForestRegressor(**rf_params))
        ],
        final_estimator= LinearRegression(),
        cv = 5,
        n_jobs=-1,
        passthrough=False
    )

pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", stack)
    ])

model_pipe = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

In [36]:
scores = cross_validate(
            model_pipe,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

In [43]:
scores['test_mae'].mean()

np.float64(-3.0238586954353854)

In [37]:
model_pipe.fit(X_train,y_train)

y_pred_train = model_pipe.predict(X_train)
y_pred_test = model_pipe.predict(X_test)

train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


In [39]:
train_r2

0.8762726273587268

In [40]:
test_r2

0.8404692715530279

In [41]:
train_mae

2.674158233088662

In [42]:
test_mae

2.981896146478075

In [44]:
mlflow.set_experiment('Exp 6 : Final Stacking Regressor')

artifact_dir = Path.cwd()
best_xgb_params_path = artifact_dir / 'xgb_best_params.json'
best_rf_params_path = artifact_dir / 'rf_best_params.json' 

with mlflow.start_run(run_name='Final Stacking Regressor') as run:
    mlflow.set_tags({
        "objective_metric": "mae",
        "model_type": "Stacking Regressor",
        "meta_model": "Linear Regression",
        "base_models": "XGBoost, Random Forest",
        "created_by": "Avanindra Bose"
    })

    preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
    )

    stack = StackingRegressor(
        estimators=[
            ("xgb", XGBRegressor(**xgb_params)),
            ("rf", RandomForestRegressor(**rf_params))
        ],
        final_estimator= LinearRegression(),
        cv = 5,
        n_jobs=-1,
        passthrough=False
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", stack)
    ])

    model_pipe = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
        )
    
    model_pipe.fit(X_train,y_train)

    scores = cross_validate(
            model_pipe,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

    y_pred_train = model_pipe.predict(X_train)
    y_pred_test = model_pipe.predict(X_test)

    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    val_mae = -scores["test_mae"].mean()

    mlflow.log_metrics({
        "best_cv_mae": val_mae,
        "final_train_mae": train_mae,
        "final_test_mae": test_mae,
        "final_train_r2": train_r2,
        "final_test_r2": test_r2,
    })

    mlflow.log_artifact(str(best_xgb_params_path), artifact_path='xgb_best_params.json')
    mlflow.log_artifact(str(best_rf_params_path), artifact_path='rf_best_params.json')

    mlflow.sklearn.log_model(
    sk_model=model_pipe,
    name="final_stacking_regresor",
    input_example=X_train.iloc[:5],          
    signature=mlflow.models.infer_signature( 
        X_train, 
        model_pipe.predict(X_train)
        )
    )    

2026/06/05 02:38:36 INFO mlflow.tracking.fluent: Experiment with name 'Exp 6 : Final Stacking Regressor' does not exist. Creating a new experiment.
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (tra

🏃 View run Final Stacking Regressor at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/8/runs/709a2c95de2b4af8b90e628f216f06a9
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/8
